# Predire l'imprevedibile — Demo introduttiva

**Obiettivo della demo**: mostrare i primi passi operativi del progetto di rilevamento anomalie nel traffico di Bologna.

In questo notebook vediamo solo le fasi preliminari:
1. **Scaricamento** dei dati di traffico (spire Comune di Bologna) e dei dati meteo (Open-Meteo)
2. **Merge** dei due dataset su timestamp
3. **Visualizzazioni di base** per esplorare i dati

> Questa è una demo veloce: lavoriamo su **poche spire** e su un **periodo breve** (un mese) per non appesantire la rete e le elaborazioni.

## 0. Setup

In [ ]:
# =============================================================================
# SETUP — Import delle librerie e parametri globali della demo
# =============================================================================
# Stack data science classico:
#   - pandas / numpy : manipolazione tabellare e operazioni numeriche
#   - requests       : chiamate HTTP alle API Open Data del Comune e Open-Meteo
#   - matplotlib     : engine di plotting di base
#   - seaborn        : stile e grafici statistici di alto livello (heatmap, box)
#   - pathlib.Path   : gestione path cross-platform per la cache locale
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Stile uniforme per tutti i grafici del notebook.
# 'whitegrid' rende le griglie ben leggibili sui plot di serie storiche.
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

# Cartella di cache locale: tutti i dati scaricati vengono salvati qui in
# formato Parquet. L'idea (in linea con la traccia, sezione 4 "Vincoli pratici")
# è scaricare UNA volta sola, poi lavorare offline ad ogni rerun.
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

# -----------------------------------------------------------------------------
# Parametri della demo
# -----------------------------------------------------------------------------
# Scelte deliberatamente "minimali" per tenere veloci download ed elaborazioni
# durante la dimostrazione. Nel progetto vero questi valori vanno ampliati:
#   - finestra temporale consigliata: gennaio 2024 → settembre 2025
#   - numero di spire: 15-25 selezionate per copertura geografica/funzionale
ANNO = 2024
DATA_INIZIO = '2024-06-01'
DATA_FINE   = '2024-06-30'
N_SPIRE_DEMO = 5    # numero di spire da campionare (top per numero rilevazioni)

# Coordinate di Bologna usate per la chiamata all'API meteo storica.
# Approssimazione del centro città: in città il meteo è omogeneo, quindi
# un singolo punto è sufficiente come proxy per tutte le spire.
LAT, LON = 44.49, 11.34

print(f'Periodo demo: {DATA_INIZIO} → {DATA_FINE}')
print(f'Numero spire selezionate: {N_SPIRE_DEMO}')

## 1. Scaricamento dati traffico (spire Comune di Bologna)

I dati sono pubblicati sul portale Open Data del Comune di Bologna. Usiamo l'API Opendatasoft con un filtro `where` sulla data per scaricare solo il periodo che ci interessa.

**Schema del dato grezzo**: ogni riga = una spira in un giorno specifico, con 24 colonne orarie (`00:00-01:00`, `01:00-02:00`, ..., `23:00-24:00`). Faremo un **melt** per portare il dataset in formato *long* (una riga per timestamp orario).

In [ ]:
def scarica_spire(anno: int, data_inizio: str, data_fine: str) -> pd.DataFrame:
    """Scarica i flussi orari delle spire di Bologna per un intervallo di date.

    Parametri
    ---------
    anno : int
        Anno di riferimento. Determina il dataset Opendatasoft da interrogare
        (esiste un dataset distinto per ogni anno dal 2019 al 2025).
    data_inizio, data_fine : str
        Estremi inclusivi dell'intervallo, in formato ISO 'YYYY-MM-DD'.
        Vengono passati come filtro `where` lato server per ridurre la
        dimensione della risposta JSON.

    Note implementative
    -------------------
    Usiamo l'endpoint `/exports/json` invece del classico `/records` perché
    quest'ultimo è paginato con limite massimo offset=10000: per finestre
    temporali ampie (e Bologna ha decine di spire × 24 ore × N giorni) si
    incappa subito nel limite. `/exports` invece restituisce in un colpo
    solo tutti i record che matchano il filtro.

    Ritorna
    -------
    pd.DataFrame
        Formato "wide": una riga per coppia (spira, giorno), 24 colonne
        orarie `00_00_01_00`, `01_00_02_00`, ..., `23_00_24_00` con il
        conteggio veicoli per ciascuna fascia.
    """
    # ID del dataset annuale (pattern documentato dal portale Open Data).
    dataset_id = f'rilevazione-flusso-veicoli-tramite-spire-anno-{anno}'
    url = f'https://opendata.comune.bologna.it/api/explore/v2.1/catalog/datasets/{dataset_id}/exports/json'

    # Filtro server-side: limita i record alla finestra richiesta.
    # NB: il nome del campo è 'data' (non 'date'): è il campo del record originale.
    params = {
        "where": f"data >= '{data_inizio}' AND data <= '{data_fine}'",
    }

    # Timeout alto (120s) perché la risposta può essere di diversi MB
    # per finestre temporali significative.
    r = requests.get(url, params=params, timeout=120)
    r.raise_for_status()  # solleva eccezione su 4xx/5xx
    return pd.DataFrame(r.json())


# -----------------------------------------------------------------------------
# Caching su filesystem: pattern "read-through cache"
# -----------------------------------------------------------------------------
# La prima esecuzione scarica e salva su Parquet; le successive leggono
# direttamente dal file locale. Parquet è preferito a CSV perché:
#   - mantiene i dtype (date, numerici, stringhe categoriche)
#   - è compresso (~10x meno disco rispetto a CSV)
#   - I/O molto più veloce
file_spire = DATA_DIR / f'spire_{DATA_INIZIO}_{DATA_FINE}.parquet'
if file_spire.exists():
    df_spire_raw = pd.read_parquet(file_spire)
    print(f'Letto da cache: {file_spire}')
else:
    df_spire_raw = scarica_spire(ANNO, DATA_INIZIO, DATA_FINE)
    df_spire_raw.to_parquet(file_spire)
    print(f'Scaricato e salvato: {file_spire}')

print(f'Shape grezzo: {df_spire_raw.shape}')
df_spire_raw.head(3)


In [ ]:
# Ispezione delle colonne disponibili nel dataset grezzo.
# Servono a confermare la struttura attesa:
#   - 'data'                              : data della rilevazione
#   - '00_00_01_00' ... '23_00_24_00'     : 24 colonne orarie (formato wide)
#   - 'id_uni'                            : ID logico della stazione (multi-direzione)
#   - 'chiave'                            : ID fisico del singolo sensore
#   - 'codice_spira'                      : codice "tecnico", può cambiare nel tempo
#   - 'nome_via', 'nodo_da', 'nodo_a'     : metadati di localizzazione
#   - 'direzione', 'angolo'               : orientamento della rilevazione
#   - 'longitudine', 'latitudine'         : coordinate geografiche
#   - 'num_giorno_settimana'              : etichetta del giorno (Lunedì, ...)
print('Colonne:', list(df_spire_raw.columns))

### Selezione di poche spire e melt in formato long

Selezioniamo solo `N_SPIRE_DEMO` spire (quelle con il maggior numero di rilevazioni nel periodo) e portiamo i dati in formato *long*: una riga = `(spira, timestamp orario, conteggio)`.

In [ ]:
# =============================================================================
# SELEZIONE SPIRE + RIDUZIONE A FORMATO "LONG"
# =============================================================================
# Obiettivo: dal dataset wide (24 colonne orarie) ricavare una serie storica
# vera e propria, con una riga per ogni (spira, timestamp orario).
# Questo è il passaggio di preprocessing chiave indicato nella traccia (sez. 3.1).
# -----------------------------------------------------------------------------

# 1) ---- Selezione delle "top N" spire del periodo ---------------------------
#    Strategia: prendiamo le N spire con più rilevazioni (= meno gap) nel
#    periodo, in modo da partire dai sensori più affidabili per la demo.
#
#    PERCHÉ raggruppiamo per (id_uni, chiave) e non solo per id_uni?
#    In alcuni casi lo stesso `id_uni` (stazione logica) copre più sensori
#    fisici distinti — tipicamente le due direzioni di marcia sulla stessa
#    sezione stradale. Selezionare solo per `id_uni` produrrebbe record
#    duplicati al successivo merge con il dataset di accuratezza
#    (che invece è indicizzato per `chiave`, l'ID del singolo sensore).
top_spire = (
    df_spire_raw.groupby(['id_uni', 'chiave']).size()
      .sort_values(ascending=False)
      .head(N_SPIRE_DEMO)
      .index.tolist()  # lista di tuple (id_uni, chiave)
)
print('Spire selezionate (id_uni, chiave):', top_spire)

# Filtraggio del DataFrame originale tramite MultiIndex.
# Tipico pattern pandas: `set_index(...).index.isin(...)` permette di
# applicare un filtro su una combinazione di colonne.
mask = df_spire_raw.set_index(['id_uni', 'chiave']).index.isin(top_spire)
df_sel = df_spire_raw[mask].copy()

# 2) ---- Identificazione delle 24 colonne orarie -----------------------------
#    Le colonne orarie del dataset flussi hanno il pattern `HH_00_HH_00`
#    (4 segmenti di 2 cifre). Usiamo una regex per individuarle in modo
#    robusto, evitando di hardcodare la lista.
#
#    ATTENZIONE: il dataset accuratezza usa invece il pattern `HH_00_HH`
#    (3 segmenti) — gestito separatamente più avanti.
import re
pat = re.compile(r'^\d{2}_\d{2}_\d{2}_\d{2}$')
colonne_orarie = [c for c in df_sel.columns if pat.match(c)]
print(f'Colonne orarie trovate: {len(colonne_orarie)}')

# 3) ---- Melt: da wide a long ------------------------------------------------
#    `pd.DataFrame.melt` "ribalta" le 24 colonne orarie in 2 colonne:
#      - `fascia_oraria`     : nome originario della colonna (es. '08_00_09_00')
#      - `conteggio_veicoli` : valore numerico
#    Le id_vars sono le colonne che NON vengono "fuse" (rimangono replicate
#    per ogni riga del long).
id_vars = ['data', 'id_uni', 'chiave']
for opt in ['nome_via', 'direzione', 'longitudine', 'latitudine']:
    # Aggiunte opzionalmente: presenti nel dataset corrente ma non garantite
    # per tutte le annate (lo schema è leggermente evoluto).
    if opt in df_sel.columns:
        id_vars.append(opt)

df_long = df_sel.melt(
    id_vars=id_vars,
    value_vars=colonne_orarie,
    var_name='fascia_oraria',
    value_name='conteggio_veicoli',
)

# 4) ---- Costruzione del timestamp orario -----------------------------------
#    Dal nome '08_00_09_00' estraiamo solo l'ora di inizio (i primi 2 char)
#    e la sommiamo come Timedelta alla colonna `data`. Risultato: un
#    pd.Timestamp per ciascuna (spira, ora) — l'unità di osservazione corretta
#    per qualsiasi analisi di serie storica.
df_long['ora'] = df_long['fascia_oraria'].str[:2].astype(int)
df_long['timestamp'] = pd.to_datetime(df_long['data']) + pd.to_timedelta(df_long['ora'], unit='h')

# Coercizione a numerico: se Opendatasoft restituisce stringhe o valori
# anomali, `errors='coerce'` li trasforma in NaN invece di sollevare errore.
df_long['conteggio_veicoli'] = pd.to_numeric(df_long['conteggio_veicoli'], errors='coerce')

# Ordinamento per (sensore, tempo) — utile per qualsiasi feature engineering
# basato su lag (lag 1h, 24h, 168h = settimana).
df_long = df_long.sort_values(['chiave', 'timestamp']).reset_index(drop=True)
print('')
print(f'Shape dataset long: {df_long.shape}')
df_long[['timestamp', 'id_uni', 'chiave', 'nome_via', 'conteggio_veicoli']].head()


## 2. Scaricamento dati meteo (Open-Meteo)

Open-Meteo offre un endpoint storico gratuito senza chiave. Scarichiamo temperatura, precipitazioni e velocità del vento orarie per Bologna nello stesso periodo.

In [ ]:
def scarica_meteo(lat: float, lon: float, data_inizio: str, data_fine: str) -> pd.DataFrame:
    """Scarica i dati meteo orari da Open-Meteo per una località e un periodo.

    L'API `archive-api.open-meteo.com` è pubblica, gratuita e senza chiave;
    copre dati storici dal 1940 in poi con granularità oraria.

    Parametri
    ---------
    lat, lon : float
        Latitudine e longitudine della località. Per Bologna ≈ (44.49, 11.34).
    data_inizio, data_fine : str
        Estremi inclusivi in formato 'YYYY-MM-DD'.

    Variabili meteo richieste
    -------------------------
    - temperature_2m  : temperatura a 2 metri dal suolo (°C)
    - precipitation   : precipitazione totale oraria (mm) — pioggia + neve
    - rain            : solo componente pioggia (mm)
    - wind_speed_10m  : velocità vento a 10 m (km/h)
    - weather_code    : codice WMO qualitativo (sereno, pioggia, neve, ...)

    Fuso orario
    -----------
    `timezone='Europe/Rome'`: CRUCIALE. La traccia (sez. 8, "Trappole comuni")
    avverte esplicitamente che Open-Meteo di default usa UTC, mentre i dati
    delle spire bolognesi sono in ora locale Europa/Roma. Un offset di 1-2 ore
    sui campi meteo rende il join sostanzialmente inutile per il modeling.

    Ritorna
    -------
    pd.DataFrame
        Una riga per ora del periodo richiesto. Indice numerico; timestamp
        come colonna pd.Timestamp (tz-naive — interpretata come ora locale).
    """
    url = 'https://archive-api.open-meteo.com/v1/archive'
    params = {
        'latitude': lat,
        'longitude': lon,
        'start_date': data_inizio,
        'end_date': data_fine,
        # Stringa CSV: lista delle variabili orarie da includere nella risposta.
        'hourly': 'temperature_2m,precipitation,rain,wind_speed_10m,weather_code',
        'timezone': 'Europe/Rome',  # vedi docstring per il "perché"
    }
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()

    # Struttura della risposta JSON di Open-Meteo:
    #   { "hourly": { "time": [...], "temperature_2m": [...], "rain": [...], ... } }
    # Ogni chiave è una lista parallela: la prima posizione di ognuna
    # corrisponde alla prima ora, la seconda alla seconda, etc.
    h = r.json()['hourly']
    df = pd.DataFrame(h)

    # 'time' arriva come stringa ISO ('2024-06-01T00:00'). Lo convertiamo in
    # pd.Timestamp e lo rinominiamo `timestamp` per uniformità con il df spire.
    df['timestamp'] = pd.to_datetime(df['time'])
    return df.drop(columns=['time'])


# -----------------------------------------------------------------------------
# Stesso pattern di caching usato per le spire: read-through cache su Parquet.
# -----------------------------------------------------------------------------
file_meteo = DATA_DIR / f'meteo_{DATA_INIZIO}_{DATA_FINE}.parquet'
if file_meteo.exists():
    df_meteo = pd.read_parquet(file_meteo)
    print(f'Letto da cache: {file_meteo}')
else:
    df_meteo = scarica_meteo(LAT, LON, DATA_INIZIO, DATA_FINE)
    df_meteo.to_parquet(file_meteo)
    print(f'Scaricato e salvato: {file_meteo}')

print(f'Shape meteo: {df_meteo.shape}')
df_meteo.head()

### Tabella di mapping `weather_code` (WMO)

Open-Meteo restituisce un codice numerico WMO. Convertiamolo in etichette qualitative leggibili: utile sia per i grafici sia per chi userà i dati nel resto del progetto.

Riferimento: [WMO Weather interpretation codes](https://open-meteo.com/en/docs/historical-weather-api) (sezione *WMO Weather interpretation codes*).


In [ ]:
# =============================================================================
# DECODIFICA CODICI WMO -> ETICHETTE QUALITATIVE LEGGIBILI
# =============================================================================
# `weather_code` di Open-Meteo è un intero che segue lo standard WMO 4677/4680
# (World Meteorological Organization). Sono valori non contigui (0, 1, 2, 3,
# 45, 48, 51, ...): l'intervallo non ha significato ordinale puro, quindi una
# mappatura esplicita è il modo corretto di gestirli.
#
# Uso nel progetto:
#   - sui grafici, usiamo l'etichetta testuale (più leggibile)
#   - in modeling, il `weather_code` può essere usato come feature categorica
#     (one-hot encoding) oppure raggruppato in macro-categorie
#     (sereno / nuvoloso / pioggia / neve / temporale).
#
# Fonte: https://open-meteo.com/en/docs/historical-weather-api
# -----------------------------------------------------------------------------
WMO_MAP = {
    # Cielo sereno / nuvolosità
    0:  'Sereno',
    1:  'Prevalentemente sereno',
    2:  'Parzialmente nuvoloso',
    3:  'Coperto',
    # Nebbia
    45: 'Nebbia',
    48: 'Nebbia con brina',
    # Pioggerella (drizzle)
    51: 'Pioggerella leggera',
    53: 'Pioggerella moderata',
    55: 'Pioggerella intensa',
    56: 'Pioggerella gelata leggera',
    57: 'Pioggerella gelata intensa',
    # Pioggia
    61: 'Pioggia leggera',
    63: 'Pioggia moderata',
    65: 'Pioggia intensa',
    66: 'Pioggia gelata leggera',
    67: 'Pioggia gelata intensa',
    # Neve
    71: 'Neve leggera',
    73: 'Neve moderata',
    75: 'Neve intensa',
    77: 'Granuli di neve',
    # Rovesci (showers)
    80: 'Rovesci leggeri',
    81: 'Rovesci moderati',
    82: 'Rovesci violenti',
    85: 'Rovesci di neve leggeri',
    86: 'Rovesci di neve intensi',
    # Temporali
    95: 'Temporale',
    96: 'Temporale con grandine leggera',
    99: 'Temporale con grandine intensa',
}

# Applicazione della mappa con fallback 'Sconosciuto'.
# `Series.map(dict)` ritorna NaN per le chiavi non presenti — `fillna` cattura
# eventuali codici futuri non ancora mappati senza far crashare il pipeline.
df_meteo['tempo'] = df_meteo['weather_code'].map(WMO_MAP).fillna('Sconosciuto')

# Sanity check: distribuzione delle condizioni meteo nel periodo della demo.
df_meteo['tempo'].value_counts()


## 3. Merge dei due dataset

Uniamo traffico e meteo sul `timestamp`. Il meteo è uno solo per tutta la città, quindi viene replicato su tutte le spire (left join sul traffico).

In [ ]:
# =============================================================================
# JOIN TRAFFICO ↔ METEO + DERIVAZIONE FEATURE TEMPORALI BASE
# =============================================================================
# - LEFT JOIN su `timestamp`: vogliamo conservare tutte le righe del traffico,
#   anche se per qualche motivo dovesse mancare il meteo in quell'ora.
# - Cardinalità: il meteo è "uno per città" (un singolo punto di osservazione),
#   quindi viene replicato/broadcast su tutte le spire dello stesso timestamp.
#   Questo è coerente con l'assunzione che la pioggia/temperatura cambino
#   uniformemente nell'area cittadina di Bologna.
# -----------------------------------------------------------------------------
df = df_long.merge(df_meteo, on='timestamp', how='left')

# -----------------------------------------------------------------------------
# Feature temporali ausiliarie per analisi e grafici.
# Sono "gratis" da derivare ma utilissime per:
#   - segmentare i grafici per giorno settimana (profili tipici)
#   - costruire feature periodiche nei modelli di forecasting
#   - distinguere weekend (cambio di regime di traffico)
# -----------------------------------------------------------------------------
df['giorno_settimana'] = df['timestamp'].dt.day_name()        # 'Monday', 'Tuesday', ...
df['ora_del_giorno']   = df['timestamp'].dt.hour              # 0..23
df['weekend']          = df['timestamp'].dt.dayofweek >= 5    # True = Sab/Dom

print(f'Shape dataset finale: {df.shape}')
print(f'Periodo: {df["timestamp"].min()} → {df["timestamp"].max()}')
print(f'Spire (chiavi distinte): {df["chiave"].nunique()}')
df.head()

## 4. Visualizzazioni di base

Quattro grafici introduttivi per farci un'idea della struttura dei dati.

### 4.1 Serie storica del flusso, una spira

In [ ]:
# =============================================================================
# VIZ 4.1 — Serie storica del flusso per UNA singola spira
# =============================================================================
# Mostra il "polso" del traffico nel mese: utile per cogliere a colpo d'occhio
# la doppia stagionalità (ciclo giornaliero a 24h + ciclo settimanale a 7d)
# e individuare visivamente eventuali outlier evidenti (drop a zero, picchi).
# -----------------------------------------------------------------------------

# Selezioniamo la PRIMA spira nella lista delle top_spire (quella col maggior
# numero di rilevazioni). Estraiamo entrambi gli identificativi:
#   id_uni : stazione logica
#   chiave : sensore fisico (necessario perché stessa stazione può avere più sensori)
id_uni_demo, chiave_demo = top_spire[0]

# Filtraggio doppio (id_uni AND chiave) per isolare ESATTAMENTE quel sensore.
# `sort_values('timestamp')` garantisce che la linea del plot sia disegnata
# in ordine cronologico (cruciale per i line plot di serie storiche).
df_one = (
    df[(df['id_uni'] == id_uni_demo) & (df['chiave'] == chiave_demo)]
      .sort_values('timestamp')
)

# Etichetta del titolo: usiamo il nome della via se disponibile, fallback su id.
via = df_one['nome_via'].iloc[0] if 'nome_via' in df_one.columns else id_uni_demo

# Linewidth=0.8 è una scelta deliberata: in un mese ci sono 30*24=720 punti,
# una linea spessa renderebbe il grafico illeggibile.
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df_one['timestamp'], df_one['conteggio_veicoli'], linewidth=0.8)
ax.set_title(f'Flusso orario - id_uni {id_uni_demo} / chiave {chiave_demo} ({via})')
ax.set_xlabel('Tempo')
ax.set_ylabel('Veicoli/ora')
plt.tight_layout()
plt.show()


### 4.2 Profilo orario medio per giorno della settimana

Il classico pattern feriale (doppia gobba ore di punta) vs. weekend.

In [ ]:
# =============================================================================
# VIZ 4.2 — Profilo orario medio per giorno della settimana
# =============================================================================
# Risponde alla domanda di EDA (traccia, sez. 2.a): "Come si comporta il
# traffico nelle 24 ore di un giorno medio, separando per giorno della
# settimana?". È il grafico più informativo per impostare modelli con
# stagionalità multipla (settimanale × giornaliera).
#
# Pattern atteso:
#   - Lun-Ven: "doppia gobba" (picchi 7-9 e 17-19, ore di punta)
#   - Sab    : picco serale spostato, gobba mattutina più bassa
#   - Dom    : profilo schiacciato, picco unico nel pomeriggio
# Discrepanze grosse da questo pattern → candidati anomalia contestuale.
# -----------------------------------------------------------------------------

# Ordine esplicito dei giorni: pandas di default ordina alfabeticamente
# ('Friday' viene prima di 'Monday'), che è inutile per la lettura.
ordine_giorni = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Group-by su due chiavi: media del flusso per ogni combinazione
# (giorno_settimana, ora_del_giorno). Risultato: una tabella long con 7*24=168 righe.
profilo = (
    df.groupby(['giorno_settimana', 'ora_del_giorno'])['conteggio_veicoli']
      .mean()
      .reset_index()
)

# Plot: una linea per ciascun giorno della settimana sull'asse x = ora.
# Iteriamo seguendo l'ordine cronologico per coerenza visiva nella legenda.
fig, ax = plt.subplots(figsize=(12, 5))
for g in ordine_giorni:
    sub = profilo[profilo['giorno_settimana'] == g]
    ax.plot(sub['ora_del_giorno'], sub['conteggio_veicoli'], marker='o', label=g)
ax.set_title('Profilo orario medio per giorno della settimana')
ax.set_xlabel('Ora del giorno')
ax.set_ylabel('Veicoli/ora (media su tutte le spire)')
ax.set_xticks(range(0, 24))           # tick orari espliciti per leggibilità
ax.legend(ncol=4, fontsize=9)         # legenda compatta a 4 colonne
plt.tight_layout()
plt.show()

### 4.3 Heatmap traffico (ora × giorno)

Una vista d'insieme: quanto traffico passa, mediamente, in ogni cella `(giorno × ora)`.

In [ ]:
# =============================================================================
# VIZ 4.3 — Heatmap "giorno settimana × ora": vista d'insieme della settimana tipo
# =============================================================================
# Stessa informazione del profilo 4.2 ma compressa in una matrice 7×24:
# permette di "vedere" simultaneamente la stagionalità settimanale e quella
# giornaliera. È particolarmente efficace per spotare:
#   - asimmetrie tra mattina/sera nei diversi giorni
#   - giorni "anomali" (es. un sabato che sembra un mercoledì → manifestazione?)
#   - spire con pattern irregolari rispetto al resto della rete
# -----------------------------------------------------------------------------

# `unstack()` trasforma il MultiIndex (giorno, ora) in una matrice 2D dove
# le righe sono giorni e le colonne sono ore — formato ideale per heatmap.
# `.reindex(ordine_giorni)` forza l'ordine Lun→Dom (vedi nota sulla cella 4.2).
pivot = (
    df.groupby(['giorno_settimana', 'ora_del_giorno'])['conteggio_veicoli']
      .mean()
      .unstack()
      .reindex(ordine_giorni)
)

# Heatmap con seaborn:
#   - cmap='viridis' : palette percettivamente uniforme (raccomandata)
#   - cbar_kws       : etichetta esplicita sulla colorbar per unità di misura
fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(pivot, cmap='viridis', cbar_kws={'label': 'Veicoli/ora'}, ax=ax)
ax.set_title('Heatmap traffico medio — giorno della settimana × ora')
ax.set_xlabel('Ora del giorno')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

### 4.4 Traffico vs. pioggia

Un primo sguardo all'effetto del meteo: il flusso cambia quando piove?

In [ ]:
# =============================================================================
# VIZ 4.4 — Effetto del meteo sul traffico (boxplot + scatter)
# =============================================================================
# Risponde all'altra domanda EDA della traccia: "Quanto e come il meteo
# influenza il traffico?". Due viste complementari:
#   (a) boxplot binario   : confronto distribuzionale piove vs asciutto
#   (b) scatter continuo : flusso vs temperatura, colorato per pioggia
# Importante: l'effetto del meteo dipende dalla tipologia di strada e
# dall'ora — questo grafico aggregato dà un'idea grossolana, non la verità.
# -----------------------------------------------------------------------------

# Variabile binaria di comodo: "piove davvero" se precipitazione > 0.1 mm/h.
# Soglia 0.1 mm è convenzionale (scarta tracce trascurabili). fillna(0)
# protegge da NaN sparsi nel meteo.
df['pioggia'] = df['precipitation'].fillna(0) > 0.1

# Layout a 2 pannelli (1 riga × 2 colonne).
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# --- (a) Boxplot: distribuzione del flusso con/senza pioggia ----------------
# Il boxplot mostra mediana, quartili e outlier per ciascun gruppo —
# molto più informativo della semplice media per distribuzioni asimmetriche.
sns.boxplot(data=df, x='pioggia', y='conteggio_veicoli', ax=axes[0])
axes[0].set_title('Distribuzione del flusso: pioggia vs. asciutto')
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['Asciutto', 'Pioggia'])
axes[0].set_xlabel('')
axes[0].set_ylabel('Veicoli/ora')

# --- (b) Scatter: temperatura vs flusso aggregato per ora -------------------
# Aggregazione preventiva: per ogni `timestamp` riduciamo le N spire a un
# unico valore medio. Questo evita di sovrappopolare lo scatter (e dà uno
# scatter dove "ogni punto = un'ora di Bologna").
# `first` su temperatura/pioggia perché sono uguali per tutte le spire
# nello stesso timestamp (già replicate dal merge precedente).
agg = df.groupby('timestamp').agg(
    flusso=('conteggio_veicoli', 'mean'),
    temperatura=('temperature_2m', 'first'),
    pioggia=('precipitation', 'first'),
).reset_index()

# `hue='pioggia'` colora i punti per quantità di pioggia in mm — fa emergere
# eventuali pattern del tipo "le ore piovose tendono a essere a sx/dx".
sns.scatterplot(data=agg, x='temperatura', y='flusso', hue='pioggia',
                palette='viridis', s=20, ax=axes[1])
axes[1].set_title('Flusso medio orario vs. temperatura')
axes[1].set_xlabel('Temperatura (°C)')
axes[1].set_ylabel('Veicoli/ora (media spire)')

plt.tight_layout()
plt.show()

## 5. Dataset accuratezza spire

Il documento di progetto è chiaro: questo dataset *non è un dettaglio tecnico, è il cuore del progetto*. Permette di distinguere una spira rotta (accuratezza 0%) da una strada davvero vuota (accuratezza 100% con conteggio 0).

Schema simile al flusso ma con qualche differenza:
- chiavi rinominate (`data_2`, `codice_spira_2`)
- colonne orarie in formato `HH_00_HH` (3 segmenti invece di 4)
- valori in formato stringa `'85%'` da convertire in numerico
- per fare il join con il dataset principale si usa il campo `chiave` (presente in entrambi)


In [ ]:
def scarica_accuratezza(anno: int, data_inizio: str, data_fine: str) -> pd.DataFrame:
    """Scarica il dataset di accuratezza orario delle spire di Bologna.

    Esiste un dataset "gemello" per ogni anno di flussi, che riporta
    l'accuratezza percentuale della rilevazione per ciascuna fascia oraria.

    Semantica dell'accuratezza (vedi traccia, sez. 3.2)
    ----------------------------------------------------
    Per la coppia (spira, ora):
      100% = rilevazione corretta per tutti i 60 minuti
      0%   = nessuna rilevazione nell'ora
      x%   = rilevazione parziale (es. 50% = il sensore ha funzionato per ~30 min)

    Questo dataset è la chiave per distinguere:
      - anomalia del FENOMENO (es. strada chiusa per emergenza): accuratezza
        alta MA conteggio basso/zero → questo è ciò che il committente vuole
      - anomalia dello STRUMENTO (sensore guasto): accuratezza bassa → da
        gestire ma NON segnalare come anomalia di traffico

    Schema specifico (diverso dal dataset flussi)
    ----------------------------------------------
      - campo data:        'data_2'             (non 'data')
      - campo spira:       'codice_spira_2'     (non 'codice_spira')
      - colonne orarie:    'HH_00_HH'           (3 segmenti, non 4)
      - valori:            stringhe '85%' (con simbolo) — vanno parsati
      - sentinel:          '-1%' = dato mancante (vedi gestione in cella successiva)
      - join key con flussi: il campo 'chiave' è presente in entrambi i dataset
    """
    dataset_id = f'accuratezza-spire-anno-{anno}'
    url = f'https://opendata.comune.bologna.it/api/explore/v2.1/catalog/datasets/{dataset_id}/exports/json'
    # NB: il nome del campo data qui è 'data_2', NON 'data' come nel dataset flussi.
    params = {'where': f"data_2 >= '{data_inizio}' AND data_2 <= '{data_fine}'"}
    r = requests.get(url, params=params, timeout=120)
    r.raise_for_status()
    return pd.DataFrame(r.json())


# Stesso pattern di cache su filesystem usato per gli altri dataset.
file_acc = DATA_DIR / f'accuratezza_{DATA_INIZIO}_{DATA_FINE}.parquet'
if file_acc.exists():
    df_acc_raw = pd.read_parquet(file_acc)
    print(f'Letto da cache: {file_acc}')
else:
    df_acc_raw = scarica_accuratezza(ANNO, DATA_INIZIO, DATA_FINE)
    df_acc_raw.to_parquet(file_acc)
    print(f'Scaricato e salvato: {file_acc}')

print(f'Shape grezzo accuratezza: {df_acc_raw.shape}')
df_acc_raw.head(2)


In [ ]:
# =============================================================================
# PREPROCESSING ACCURATEZZA: filter + melt + parsing percentuali
# =============================================================================
# Stesso pattern del dataset flussi (wide → long) ma con tre differenze
# di schema che gestiamo qui esplicitamente.
# -----------------------------------------------------------------------------

# 1) ---- Filtro alle stesse spire selezionate per i flussi ------------------
# La join key naturale tra i due dataset è il campo `chiave` (ID del sensore
# fisico). Estraiamo l'elenco delle chiavi presenti nel df finale dei flussi.
chiavi_demo = df['chiave'].dropna().unique().tolist()
print(f'Chiavi accuratezza da filtrare: {chiavi_demo}')

df_acc_sel = df_acc_raw[df_acc_raw['chiave'].isin(chiavi_demo)].copy()

# 2) ---- Identificazione delle 24 colonne orarie ---------------------------
# Differenza chiave rispetto al dataset flussi: qui il pattern è HH_00_HH
# (3 segmenti) invece di HH_00_HH_00 (4 segmenti). Es: '08_00_09' invece di
# '08_00_09_00'. La regex è quindi diversa.
pat_acc = re.compile(r'^\d{2}_\d{2}_\d{2}$')
colonne_orarie_acc = [c for c in df_acc_sel.columns if pat_acc.match(c)]
print(f'Colonne orarie accuratezza: {len(colonne_orarie_acc)}')

# 3) ---- Melt in formato long -----------------------------------------------
# id_vars: data e chiave del sensore.
# value_vars: le 24 colonne orarie.
df_acc_long = df_acc_sel.melt(
    id_vars=['data_2', 'chiave'],
    value_vars=colonne_orarie_acc,
    var_name='fascia_oraria_acc',
    value_name='accuratezza_str',
)

# 4) ---- Costruzione timestamp orario ---------------------------------------
# Identico al pattern usato per i flussi: data + Timedelta dell'ora di inizio.
df_acc_long['ora'] = df_acc_long['fascia_oraria_acc'].str[:2].astype(int)
df_acc_long['timestamp'] = (
    pd.to_datetime(df_acc_long['data_2']) + pd.to_timedelta(df_acc_long['ora'], unit='h')
)

# 5) ---- Parsing valore '85%' → 85.0 (float) -------------------------------
# Lo strip del '%' è necessario perché Opendatasoft serializza il valore come
# stringa, non come numero. Il `replace('', np.nan)` cattura eventuali stringhe
# vuote (rare ma esistenti). Il `.astype(float)` finale è l'effettiva conversione.
df_acc_long['accuratezza'] = (
    df_acc_long['accuratezza_str'].astype(str).str.rstrip('%').replace('', np.nan).astype(float)
)

# 6) ---- Gestione del valore sentinel -1 ------------------------------------
# Il dataset usa '-1%' (o anche '-2%', valori negativi) come marker di
# "dato mancante / non disponibile". Sono semanticamente diversi da 0%
# (sensore presente ma totalmente offline) e vanno convertiti in NaN per
# evitare di alterare statistiche aggregate (media, ecc.).
df_acc_long.loc[df_acc_long['accuratezza'] < 0, 'accuratezza'] = np.nan

# Riduzione alle sole colonne utili per il merge successivo.
df_acc_long = df_acc_long[['chiave', 'timestamp', 'accuratezza']]
print(f'Shape accuratezza long: {df_acc_long.shape}')
df_acc_long.head()


In [ ]:
# =============================================================================
# MERGE FINALE: traffico ↔ accuratezza sul sensore + timestamp
# =============================================================================
# Join key composito: (chiave, timestamp). La `chiave` è l'ID del sensore
# fisico (la stessa che abbiamo usato nel df flussi); il `timestamp` è già
# stato armonizzato in entrambi (datetime tz-naive ora locale).
# Left join: conserviamo tutte le righe del traffico, l'accuratezza diventa
# NaN dove mancante (sensore presente ma senza riga di accuratezza per
# quell'ora — caso raro ma possibile).
# -----------------------------------------------------------------------------
df = df.merge(df_acc_long, on=['chiave', 'timestamp'], how='left')

print(f'Shape dopo merge accuratezza: {df.shape}')
# Sanity check: tipico controllo di qualità post-merge.
# Da osservare:
#   - media: se molto < 100, le spire della demo non sono "pulitissime"
#   - min:   0 indica sensori spenti, NaN indica dati mancanti
#   - NaN:   ore senza alcuna informazione di accuratezza (problemi nel join)
print(
    f"Accuratezza: media={df['accuratezza'].mean():.1f}%  "
    f"min={df['accuratezza'].min():.0f}%  NaN={df['accuratezza'].isna().sum()}"
)
df[['timestamp', 'id_uni', 'chiave', 'conteggio_veicoli', 'accuratezza']].head()


## 6. Calendario festività italiane

Usiamo il pacchetto `holidays` (`pip install holidays`) che conosce le festività nazionali italiane. Aggiungiamo anche due colonne derivate utili per il problem framing: `tipo_giorno` (feriale / weekend / festivo) e `festa_locale` per la festa di San Petronio (4 ottobre), che `holidays` non include di default.


In [ ]:
# =============================================================================
# CALENDARIO FESTIVITÀ — Nazionali (pkg `holidays`) + locali (San Petronio)
# =============================================================================
# Le festività sono un "regime" diverso dai feriali e dai weekend, e devono
# essere trattate come tali nei modelli di anomaly detection. Esempio: un
# 25 dicembre con traffico tipico da feriale → quello SÌ è anomalo.
#
# Il pacchetto `holidays` (pip install holidays) gestisce le festività
# nazionali italiane (Capodanno, Pasqua, Festa della Repubblica, ...).
# Le festività LOCALI bolognesi (es. San Petronio il 4 ottobre) vanno
# aggiunte manualmente come indicato nella traccia (sez. 3.4).
# -----------------------------------------------------------------------------
import holidays

# Carichiamo le festività italiane per tutti gli anni presenti nei dati.
# `country_holidays('IT', years=[...])` ritorna un dict-like {data: nome}.
anni = sorted(df['timestamp'].dt.year.unique().tolist())
festivita_it = holidays.country_holidays('IT', years=anni)

# ---- Festività locali bolognesi --------------------------------------------
# Esempio minimale: solo San Petronio. In progetto vero qui andrebbe il
# "domain knowledge calendar" indicato dalla traccia: partite del Bologna FC
# in casa, fiere a BolognaFiere, manifestazioni storiche ricorrenti, ecc.
festivita_locali = {f'{a}-10-04': 'San Petronio' for a in anni}

# ---- Feature derivate sul df principale -----------------------------------
# `data_giorno` = solo la parte data (datetime.date) per match con il dict
# di `holidays` (che è indicizzato per date e non per Timestamp).
df['data_giorno']       = df['timestamp'].dt.date

# Boolean: True se il giorno è una festività nazionale italiana.
df['festivo_nazionale'] = df['data_giorno'].apply(lambda d: d in festivita_it)

# Nome della festività (es. 'Festa della Repubblica') o None.
# Utile per analisi successive: "il traffico del 2 giugno cambia di X% rispetto
# a un mercoledì normale di giugno".
df['nome_festivita']    = df['data_giorno'].apply(lambda d: festivita_it.get(d))

# Boolean: True per festività locali bolognesi (gestite con dict separato).
df['festa_locale']      = df['timestamp'].dt.strftime('%Y-%m-%d').map(festivita_locali).notna()

def categoria_giorno(row):
    """Categorizza il giorno in tre regimi distinti per il modeling.

    L'ordine dei branch è importante: la festività VINCE sul weekend
    (un Natale che cade di sabato è semanticamente più "festivo" che "weekend").
    """
    if row['festivo_nazionale'] or row['festa_locale']:
        return 'festivo'
    if row['weekend']:
        return 'weekend'
    return 'feriale'

df['tipo_giorno'] = df.apply(categoria_giorno, axis=1)

# ---- Sanity check sulla distribuzione dei regimi ---------------------------
print('Distribuzione tipi giorno (ore):')
print(df['tipo_giorno'].value_counts())
print()
print('Festività trovate nel periodo:')
fest = df.loc[df['festivo_nazionale'], ['data_giorno', 'nome_festivita']].drop_duplicates()
if len(fest):
    print(fest.to_string(index=False))
else:
    print('(nessuna nel periodo selezionato)')


## 7. Visualizzazioni con i nuovi dati

### 7.1 Heatmap accuratezza spire

Una vista d'insieme di quanto sono affidabili le misure ora-per-ora nel periodo. Le zone scure sono quelle dove il sensore ha funzionato male: è esattamente lì che il sistema di anomaly detection rischia di generare falsi positivi se non si gestisce la qualità del dato.

In [ ]:
# =============================================================================
# VIZ 7.1 — Heatmap accuratezza spire (chiave × giorno)
# =============================================================================
# Una mappa qualitativa di quanto sono "in salute" le spire selezionate.
# - Verde scuro: accuratezza ~100% (sensore affidabile)
# - Rosso scuro: accuratezza vicina a 0% (sensore guasto / in manutenzione)
#
# OPERATIVAMENTE: è qui che il problem framing della Fase 1 si traduce in
# scelte concrete. Le zone rosse della heatmap producono inevitabilmente
# falsi positivi in un sistema naive che non considera l'accuratezza.
# Tre strategie tipiche:
#   - escludere (drop)    : semplice ma riduce i dati
#   - imputare           : rischioso, può mascherare anomalie reali
#   - flaggare           : il modello vede il dato ma il sistema downstream
#                          sa che è "low confidence"
# -----------------------------------------------------------------------------

# Aggregazione: media giornaliera dell'accuratezza, per ogni spira.
# Riduciamo la granularità da oraria a giornaliera per leggibilità — una
# matrice 5 × 24 × 30 sarebbe troppo densa. `unstack()` pivota a 2D.
acc_pivot = (
    df.assign(giorno=df['timestamp'].dt.date)
      .groupby(['chiave', 'giorno'])['accuratezza']
      .mean()
      .unstack()
)

# Altezza dinamica della figura in base al numero di spire.
# Il colormap 'RdYlGn' (Red-Yellow-Green) è una scelta semantica:
#   rosso = brutto, verde = buono — leggibile anche da daltonici parziali.
# `vmin=0, vmax=100` fissa la scala alla semantica (è una percentuale),
# evitando che il colormap si "espanda" sui valori effettivamente presenti.
fig, ax = plt.subplots(figsize=(14, max(2, len(acc_pivot) * 0.5)))
sns.heatmap(acc_pivot, cmap='RdYlGn', vmin=0, vmax=100,
            cbar_kws={'label': 'Accuratezza media giornaliera (%)'}, ax=ax)
ax.set_title('Accuratezza spire - chiave x giorno')
ax.set_xlabel('Giorno')
ax.set_ylabel('chiave sensore')
plt.tight_layout()
plt.show()


### 7.2 Profilo orario: feriali vs weekend vs festivi

Aspettativa: festivi e weekend dovrebbero schiacciare la doppia gobba delle ore di punta. Se così non fosse su una determinata spira, è già un'informazione interessante per il problem framing.

In [ ]:
# =============================================================================
# VIZ 7.2 — Profilo orario per tipo di giorno (feriale / weekend / festivo)
# =============================================================================
# Riproduce il grafico 4.2 ma aggregando su un asse semanticamente più ricco:
# `tipo_giorno`, derivato dalla cella precedente. Le tre curve dovrebbero
# essere CLARAMENTE separabili — se non lo sono, vuol dire che il regime
# "festivo" del nostro periodo demo è poco rappresentato (a giugno c'è solo
# il 2/6, sample piccolo).
#
# Nel progetto vero, su una finestra di 18+ mesi, questa è UNA delle viste
# che giustifica la scelta di trattare i tre regimi come modelli separati
# (o di includere `tipo_giorno` come feature condizionante nel forecasting).
# -----------------------------------------------------------------------------

# Aggregazione: media flusso per (tipo_giorno, ora).
profilo_g = (
    df.groupby(['tipo_giorno', 'ora_del_giorno'])['conteggio_veicoli']
      .mean()
      .reset_index()
)

# Marker diversi per ciascun tipo: aiuta a distinguere le curve anche su
# stampa B/N e per chi ha difficoltà di percezione del colore.
fig, ax = plt.subplots(figsize=(12, 5))
for tipo, marker in zip(['feriale', 'weekend', 'festivo'], ['o', 's', '^']):
    sub = profilo_g[profilo_g['tipo_giorno'] == tipo]
    # `if not sub.empty`: difesa contro periodi senza alcuna festività
    # (es. solo agosto: la categoria 'festivo' potrebbe non avere righe).
    if not sub.empty:
        ax.plot(sub['ora_del_giorno'], sub['conteggio_veicoli'],
                marker=marker, label=tipo, linewidth=2)
ax.set_title('Profilo orario medio per tipo di giorno')
ax.set_xlabel('Ora del giorno')
ax.set_ylabel('Veicoli/ora (media)')
ax.set_xticks(range(0, 24))
ax.legend()
plt.tight_layout()
plt.show()


### 7.3 Tempo atmosferico → conteggio veicoli

Quale 'tipo di tempo' coincide mediamente con più o meno traffico? Attenzione: in un solo mese alcune categorie possono avere pochissime ore — interpretare con cautela.

In [ ]:
# =============================================================================
# VIZ 7.3 — Flusso medio per condizione meteo qualitativa (`tempo`)
# =============================================================================
# Aggregazione del flusso per macro-categoria meteo decodificata dal WMO code.
# Permette di rispondere a domande qualitative: "C'è meno traffico quando
# nevica? E quando c'è temporale?".
#
# CAVEAT IMPORTANTE: su un mese solo (la demo) molte categorie hanno
# pochissime osservazioni (es. solo 3 ore di 'Pioggia intensa'). La media
# è quindi RUMOROSA. La colonna `count` riportata serve proprio a far
# capire dove le medie sono robuste e dove no.
# -----------------------------------------------------------------------------

# `agg(['mean', 'count'])` ritorna due colonne in un colpo: la media
# (interesse principale) e il sample size (per giudicare l'affidabilità).
# Ordinato per media crescente: il grafico viene leggibile dall'alto in basso
# come "categorie meteo associate a meno → più traffico".
conteggio_per_tempo = (
    df.groupby('tempo')['conteggio_veicoli']
      .agg(['mean', 'count'])
      .sort_values('mean', ascending=True)
)
print(conteggio_per_tempo)

# Barh (barre orizzontali): preferite quando le etichette delle categorie
# sono testuali lunghe — leggibili senza ruotare i tick.
# Altezza dinamica: ~0.4" per ogni categoria distinta nel periodo.
fig, ax = plt.subplots(figsize=(10, max(3, 0.4 * len(conteggio_per_tempo))))
ax.barh(conteggio_per_tempo.index, conteggio_per_tempo['mean'], color='steelblue')
ax.set_xlabel('Veicoli/ora (media)')
ax.set_title('Flusso medio orario per condizione meteo')
plt.tight_layout()
plt.show()


## Prossimi passi (non in questa demo)

Quanto coperto qui — download dati, merge, accuratezza, festività, prime viz — è la base operativa. Da qui parte il progetto vero:

- **Fase 1 — Problem framing**: definizione operativa di anomalia, tipi che il sistema deve / non deve catturare, utente finale, asimmetria FP/FN.
- **Fase 2 — Tre baseline** di anomaly detection:
  - statistica classica (STL + soglia su residui via MAD)
  - density-based (IsolationForest / LOF con feature engineered)
  - forecasting-based (gradient boosting con lag + meteo come feature)
- **Fase 3 — Protocollo di valutazione** senza etichette: iniezione di anomalie sintetiche, validazione con eventi reali (15-20 eventi storici), analisi di stabilità al variare degli iperparametri, metriche custom coerenti col framing.
- **Fase 4 — Reporting** e codice riproducibile.

Tutto il dettaglio è nel documento di progetto.
